# Discrete DCP — scikit-learn, LightGBM, and quantile-forest

DCP for count data with randomized PIT, an integer quantile grid, CDF, PMF, and optimal inventory.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from quantile_forest import RandomForestQuantileRegressor
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import DiscreteDistributionalConformalPredictiveSystem
from tinyconformal.utils import MultiQuantileRegressor, NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 3, size=(3500, 1))
mu = np.exp(0.5 + 0.65 * X[:, 0])
y = rng.poisson(mu)
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)
levels = (0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99)

ModuleNotFoundError: No module named 'tinyconformal'

In [ ]:
models = {
    "scikit-learn": MultiQuantileRegressor(
        HistGradientBoostingRegressor(loss="quantile", max_iter=200, random_state=42), quantiles=levels
    ),
    "LightGBM": MultiQuantileRegressor(
        LGBMRegressor(objective="quantile", n_estimators=200, verbosity=-1, random_state=42), quantiles=levels
    ),
    "quantile-forest": RandomForestQuantileRegressor(
        n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1
    ),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    dcp = DiscreteDistributionalConformalPredictiveSystem(
        model, quantiles=levels, minimum=0, random_state=42
    ).fit(X_cal, y_cal)
    distribution = dcp.predict_distribution(X_test)
    results[name] = (dcp, distribution)
    display(distribution.evaluate(y_test))

## CDF, PMF, PPF, and quantiles

In [ ]:
distribution = results["quantile-forest"][1]
requested = np.array([0.1, 0.5, 0.9])
requested_matrix = np.broadcast_to(requested, (len(distribution), len(requested)))
quantile_predictions = distribution.ppf(requested_matrix)

pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test)[:10],
    "pmf_at_y": distribution.pmf(y_test)[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})

## Inventory solver

In [ ]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "holding_cost": 1.0,
})
stock = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
)
assert np.all(stock["y_optimal"] == np.floor(stock["y_optimal"]))
stock.head()